In [26]:
# Comments in English only
# PSRC CT saturation model (ODE in lambda) with Plotly + classes
# - RP computed over 0..2*pi (canonical RMS definition)
# - Flux plot shows lambda(t) + ie(t) (secondary y-axis)
# - Current plot shows is(t), i2(t) + 1-cycle RMS is_rms(t), i2_rms(t)
# - No Excel reading

import math
import numpy as np
import plotly.graph_objects as go


# -----------------------------
# Inputs (edit here)
# -----------------------------
inp = {
    # Simulation setup
    "f_hz": 60.0,
    "t_end": 0.25,
    "pre_fault_cycles": 1.0,
    "dt": 1.0 / 12000.0,

    # Saturation curve parameters (from excitation curve)
    "S": 22.0,          # exponent
    "Vs": 400.0,        # V rms at Ie = 10 A

    # CT and burden
    "N": 240.0,         # CT ratio 1:N (primary:secondary)
    "Rw": 0.0,          # ohm (secondary winding)
    "Rb": 4.0,          # ohm (burden resistance)
    "Xb": 2.0,          # ohm (burden reactance at f_hz)

    # System/fault
    "Ip": 12000.0,      # A rms symmetrical primary fault current
    "Off": 1.0,         # pu dc offset magnitude
    "XoverR": 12.0,     # system X/R used to compute Tau1

    # Remanence (pu of Lamsat)
    "Lamrem": 0.0       # pu
}


# -----------------------------
# Numeric helpers
# -----------------------------
def sliding_rms(x: np.ndarray, window_samples: int) -> np.ndarray:
    # Sliding RMS using a rectangular window (output same length via padding)
    if window_samples < 1:
        raise ValueError("window_samples must be >= 1")

    x2 = x * x
    w = np.ones(int(window_samples), dtype=float)

    ma_valid = np.convolve(x2, w, mode="valid") / float(window_samples)
    rms_valid = np.sqrt(ma_valid)

    if rms_valid.size == 0:
        return np.zeros_like(x, dtype=float)

    pad_len = int(window_samples - 1)
    pad = np.full(pad_len, float(rms_valid[0]), dtype=float)
    rms = np.concatenate([pad, rms_valid])
    return rms


# -----------------------------
# Model
# -----------------------------
class c:
    def __init__(self, inp_dict: dict):
        self.inp = dict(inp_dict)
        self.der = self._derived_params()

    def _compute_rp_numeric(self, S: float, n_points: int = 200000) -> float:
        # RP = Irms / Ipk for waveform proportional to |sin(theta)|^S
        # Canonical definition: integrate over 0..2*pi
        theta = np.linspace(0.0, 2.0 * math.pi, int(n_points))
        y = np.sin(theta)
        y = np.power(y, 2.0 * S)  # (sin^S)^2 = sin^(2S)
        integral = np.trapezoid(y, theta)
        irms_sq = (1.0 / (2.0 * math.pi)) * integral
        rp = math.sqrt(irms_sq)   # peak normalized to 1
        return float(rp)

    def _derived_params(self) -> dict:
        S = float(self.inp["S"])
        Vs = float(self.inp["Vs"])
        Rw = float(self.inp["Rw"])
        Rb = float(self.inp["Rb"])
        Xb = float(self.inp["Xb"])
        XoverR = float(self.inp["XoverR"])
        f_hz = float(self.inp["f_hz"])

        omega = 2.0 * math.pi * f_hz
        Rt = float(Rw + Rb)
        Tau1 = float(XoverR / omega)
        Lb = float(Xb / omega)

        RP = self._compute_rp_numeric(S=S)

        # Spreadsheet formula: A = 10*omega^S / ((Vs*sqrt(2))^S * RP)
        A = float(10.0 * (omega ** S) / (((Vs * math.sqrt(2.0)) ** S) * RP))

        # Peak flux-linkages corresponding to Vs (used for remanence scaling)
        Lamsat = float(1.41428 * Vs / omega)

        out = {}
        out["omega"] = omega
        out["Rt"] = Rt
        out["Tau1"] = Tau1
        out["Lb"] = Lb
        out["RP"] = RP
        out["A"] = A
        out["Lamsat"] = Lamsat
        return out

    def is_and_disdt(self, t: float) -> tuple[float, float]:
        # Forcing function from PSRC (eq. 15 and 16)
        Ip = float(self.inp["Ip"])
        N = float(self.inp["N"])
        Off = float(self.inp["Off"])

        omega = float(self.der["omega"])
        Tau1 = float(self.der["Tau1"])

        if t < 0.0:
            return 0.0, 0.0

        # Numeric guard for acos domain
        off_clip = float(min(0.999999, max(-0.999999, Off)))
        phi = float(math.acos(off_clip))

        term_dc = Off * math.exp(-t / Tau1)
        term_ac = math.cos((omega * t) - phi)
        is_val = math.sqrt(2.0) * Ip / N * (term_dc - term_ac)

        term_dc_d = (-Off / Tau1) * math.exp(-t / Tau1)
        term_ac_d = omega * math.sin((omega * t) - phi)
        disdt_val = math.sqrt(2.0) * Ip / N * (term_dc_d + term_ac_d)

        return float(is_val), float(disdt_val)

    def ie_of_lambda(self, lam: float) -> float:
        # ie = A*sgn(lam)*|lam|^S
        A = float(self.der["A"])
        S = float(self.inp["S"])

        if lam > 0.0:
            sgn = 1.0
        elif lam < 0.0:
            sgn = -1.0
        else:
            sgn = 0.0

        return float(A * sgn * (abs(lam) ** S))

    def die_dlambda(self, lam: float) -> float:
        # die/dlambda = A*S*|lam|^(S-1)
        A = float(self.der["A"])
        S = float(self.inp["S"])
        return float(A * S * (abs(lam) ** (S - 1.0)))

    def rhs_lambda(self, t: float, lam: float) -> float:
        # dlambda/dt * [1 + Lb * (die/dlambda)] = -Rt*ie + Rt*is + Lb*dis/dt
        Rt = float(self.der["Rt"])
        Lb = float(self.der["Lb"])

        is_val, disdt_val = self.is_and_disdt(t=t)
        ie_val = self.ie_of_lambda(lam=lam)

        denom = 1.0 + (Lb * self.die_dlambda(lam=lam))
        rhs = (-Rt * ie_val) + (Rt * is_val) + (Lb * disdt_val)

        return float(rhs / denom)

    def integrate_rk4(self, t: np.ndarray, lam0: float) -> np.ndarray:
        lam = np.zeros_like(t, dtype=float)
        lam[0] = float(lam0)

        k = 0
        while k < (t.size - 1):
            dt = float(t[k + 1] - t[k])
            tk = float(t[k])
            yk = float(lam[k])

            k1 = self.rhs_lambda(t=tk, lam=yk)
            k2 = self.rhs_lambda(t=tk + 0.5 * dt, lam=yk + 0.5 * dt * k1)
            k3 = self.rhs_lambda(t=tk + 0.5 * dt, lam=yk + 0.5 * dt * k2)
            k4 = self.rhs_lambda(t=tk + dt, lam=yk + dt * k3)

            lam[k + 1] = yk + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)
            k = k + 1

        return lam

    def simulate(self, integrator: str = "rk4") -> dict:
        f_hz = float(self.inp["f_hz"])
        t_end = float(self.inp["t_end"])
        pre_fault_cycles = float(self.inp["pre_fault_cycles"])
        dt = float(self.inp["dt"])

        t_pre = float(pre_fault_cycles / f_hz)
        t0 = -t_pre
        n_steps = int(math.floor((t_end - t0) / dt)) + 1
        t = t0 + dt * np.arange(n_steps, dtype=float)

        lam0 = float(self.der["Lamsat"] * float(self.inp["Lamrem"]) + 1e-4)

        if integrator.lower() == "rk4":
            lam = self.integrate_rk4(t=t, lam0=lam0)
        else:
            # Euler fallback
            lam = np.zeros_like(t, dtype=float)
            lam[0] = float(lam0)
            k = 0
            while k < (t.size - 1):
                dt_k = float(t[k + 1] - t[k])
                lam[k + 1] = float(lam[k] + dt_k * self.rhs_lambda(t=float(t[k]), lam=float(lam[k])))
                k = k + 1

        is_arr = np.zeros_like(t)
        disdt_arr = np.zeros_like(t)
        ie_arr = np.zeros_like(t)
        i2_arr = np.zeros_like(t)

        k = 0
        while k < t.size:
            is_val, disdt_val = self.is_and_disdt(t=float(t[k]))
            ie_val = self.ie_of_lambda(lam=float(lam[k]))

            is_arr[k] = is_val
            disdt_arr[k] = disdt_val
            ie_arr[k] = ie_val
            i2_arr[k] = float(is_val - ie_val)

            k = k + 1

        samples_per_cycle = int(round((1.0 / f_hz) / dt))
        is_rms = sliding_rms(is_arr, samples_per_cycle)
        i2_rms = sliding_rms(i2_arr, samples_per_cycle)

        out = {}
        out["t"] = t
        out["lam"] = lam
        out["is"] = is_arr
        out["ie"] = ie_arr
        out["i2"] = i2_arr
        out["disdt"] = disdt_arr
        out["is_rms"] = is_rms
        out["i2_rms"] = i2_rms
        out["samples_per_cycle"] = samples_per_cycle
        out["derived"] = dict(self.der)
        return out


# -----------------------------
# Plotting
# -----------------------------
class CTSatPlotter:
    def plot_currents(self,
                      t: np.ndarray,
                      is_arr: np.ndarray,
                      i2_arr: np.ndarray,
                      is_rms: np.ndarray,
                      i2_rms: np.ndarray) -> None:
        fig = go.Figure()

        fig.add_trace(go.Scatter(x=t, y=is_arr, mode="lines", name="is(t) ideal"))
        fig.add_trace(go.Scatter(x=t, y=i2_arr, mode="lines", name="i2(t) actual"))
        fig.add_trace(go.Scatter(x=t, y=is_rms, mode="lines", name="is_rms(t) 1-cycle"))
        fig.add_trace(go.Scatter(x=t, y=i2_rms, mode="lines", name="i2_rms(t) 1-cycle"))

        fig.update_layout(
            title="Currents (instantaneous and 1-cycle RMS)",
            xaxis_title="Time (s)",
            yaxis_title="Current (A)",
            hovermode="x unified"
        )
        fig.show()

    def plot_flux_and_excitation(self,
                                 t: np.ndarray,
                                 lam: np.ndarray,
                                 ie_arr: np.ndarray) -> None:
        fig = go.Figure()

        fig.add_trace(go.Scatter(x=t, y=lam, mode="lines", name="lambda(t)", yaxis="y1"))
        fig.add_trace(go.Scatter(x=t, y=ie_arr, mode="lines", name="ie(t) excitation", yaxis="y2"))

        fig.update_layout(
            title="Flux-linkages and excitation current",
            xaxis_title="Time (s)",
            yaxis=dict(title="Lambda (Wb-turns)", side="left"),
            yaxis2=dict(title="Excitation current ie (A)", overlaying="y", side="right"),
            hovermode="x unified"
        )
        fig.show()


# -----------------------------
# Run
# -----------------------------
model = CTSatModel(inp_dict=inp)
res = model.simulate(integrator="rk4")

print("Derived parameters:")
for key in ["omega", "Rt", "Tau1", "Lb", "RP", "A", "Lamsat"]:
    print(f"  {key}: {res['derived'][key]}")

print("Samples per cycle:", res["samples_per_cycle"])

plotter = CTSatPlotter()
plotter.plot_currents(
    t=res["t"],
    is_arr=res["is"],
    i2_arr=res["i2"],
    is_rms=res["is_rms"],
    i2_rms=res["i2_rms"]
)

# plotter.plot_flux_and_excitation(
#     t=res["t"],
#     lam=res["lam"],
#     ie_arr=res["ie"]
# )


Derived parameters:
  omega: 376.99111843077515
  Rt: 4.0
  Tau1: 0.03183098861837907
  Lb: 0.005305164769729845
  RP: 0.34583837080250085
  A: 0.003834967379587878
  Lamsat: 1.500597686106705
Samples per cycle: 200
